In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import pennylane as qml
from pennylane.qnn import TorchLayer

# ─────────────────────────────────────────────
# SEEDING
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
n_qubits     = 8           # increased to 8 for richer quantum expressivity
q_depth      = 4           # number of variational layers in quantum circuit
batch_size   = 64
num_classes  = 10
num_epochs   = 80
lr           = 0.0005      # slightly lower than pure classical due to quantum layer sensitivity
weight_decay = 1e-4

# ─────────────────────────────────────────────
# TRANSFORMS
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ─────────────────────────────────────────────
# DATASETS & LOADERS
# ─────────────────────────────────────────────
TRAIN_PATH = 'train'
TEST_PATH  = 'test'
VAL_PATH   = './val'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    print("Datasets loaded successfully")
except Exception as e:
    print(f"Error loading datasets: {e}")

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights computed:", class_weights)
except Exception as e:
    print(f"Could not compute class weights: {e}. Using uniform weights.")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# ─────────────────────────────────────────────
# QUANTUM CIRCUIT
# ─────────────────────────────────────────────
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # ── Angle Encoding ──────────────────────────────────────────
    # Each qubit gets two rotations for richer state encoding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
        qml.RZ(inputs[..., i + n_qubits], wires=i)   # using 2*n_qubits inputs

    # ── Variational Layers ──────────────────────────────────────
    for l in range(weights.shape[0]):
        # Single-qubit rotations (trainable)
        for i in range(n_qubits):
            qml.RY(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)

        # Entanglement: ring topology (proven effective for small circuits)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])   # close the ring

    # ── Measurement ─────────────────────────────────────────────
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
    # output: n_qubits values in [-1, 1]

# shape: (q_depth layers, n_qubits per layer, 2 rotations per qubit)
weight_shapes = {"weights": (q_depth, n_qubits, 2)}

# ─────────────────────────────────────────────
# BUILDING BLOCKS  (same as classical model)
# ─────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        scale = self.pool(x).view(b, c)
        scale = self.fc(scale).view(b, c, 1, 1)
        return x * scale


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.1):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se = SEBlock(out_ch)
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv_block(x)
        out = self.se(out)
        return self.relu(out + self.skip(x))


# ─────────────────────────────────────────────
# QUANTUM BRIDGE  —  maps CNN features → qubit inputs
# ─────────────────────────────────────────────
# The CNN outputs 128 features but the quantum circuit needs
# exactly 2*n_qubits (=16) angle values in [-π, π]
# This bridge compresses 128 → 16 with learned projection

class QuantumBridge(nn.Module):
    def __init__(self, in_features, n_qubits):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.LayerNorm(64),       # LayerNorm stabilizes quantum input range better than BatchNorm
            nn.GELU(),
            nn.Linear(64, n_qubits * 2)   # *2 because we use RY + RZ encoding
        )

    def forward(self, x):
        x = self.project(x)
        return torch.pi * torch.tanh(x)   # scale to (-π, π) — valid rotation angle range


# ─────────────────────────────────────────────
# MAIN MODEL  —  HybridResNet
# ─────────────────────────────────────────────
#
#  Flow:
#  Input(1,32,32)
#    → Stem         (1→32 channels)
#    → Stage1       (32→32, no downsample)
#    → Stage2       (32→64, stride=2)
#    → Stage3       (64→128, stride=2)
#    → GAP          (128,8,8 → 128)
#    → QuantumBridge (128 → 2*n_qubits angles)
#    → QuantumLayer  (angles → n_qubits expectation values)
#    → Classifier   (n_qubits → num_classes)
#
class HybridResNet(nn.Module):
    def __init__(self, n_qubits, num_classes, dropout=0.3):
        super().__init__()

        # ── Classical Backbone (same as SmallResNet) ──────────────
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.stage1 = nn.Sequential(ResBlock(32, 32),  ResBlock(32, 32))
        self.stage2 = nn.Sequential(ResBlock(32, 64, stride=2), ResBlock(64, 64))
        self.stage3 = nn.Sequential(ResBlock(64, 128, stride=2), ResBlock(128, 128))
        self.gap    = nn.AdaptiveAvgPool2d(1)

        # ── Quantum Bridge ─────────────────────────────────────────
        self.bridge = QuantumBridge(in_features=128, n_qubits=n_qubits)

        # ── Quantum Layer ──────────────────────────────────────────
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)

        # ── Classical Classifier head ──────────────────────────────
        # Input is n_qubits expectation values from [-1, 1]
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # Classical feature extraction
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)      # (B, 128)

        # Bridge: 128 → quantum angles
        x = self.bridge(x)              # (B, 2*n_qubits)

        # Quantum processing
        x = self.q_layer(x)             # (B, n_qubits) — expectation values in [-1,1]

        # Classical classification
        return self.classifier(x)


# ─────────────────────────────────────────────
# TRAINING & EVALUATION
# ─────────────────────────────────────────────
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0

    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    return total_loss / len(dataloader), correct / len(dataloader.dataset)


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / len(dataloader), correct / total


# ─────────────────────────────────────────────
# MODEL, OPTIMIZER, LOSS, SCHEDULER
# ─────────────────────────────────────────────
model = HybridResNet(n_qubits=n_qubits, num_classes=num_classes, dropout=0.3).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

# Separate LRs: classical layers train faster than quantum
optimizer = torch.optim.AdamW([
    {'params': model.stem.parameters(),       'lr': lr},
    {'params': model.stage1.parameters(),     'lr': lr},
    {'params': model.stage2.parameters(),     'lr': lr},
    {'params': model.stage3.parameters(),     'lr': lr},
    {'params': model.bridge.parameters(),     'lr': lr},
    {'params': model.q_layer.parameters(),    'lr': lr * 0.1},   # quantum layer needs slower LR
    {'params': model.classifier.parameters(), 'lr': lr},
], weight_decay=weight_decay)

loss_fn   = nn.CrossEntropyLoss(weight=class_weight_tensor, label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# ─────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────
best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs, val_accs     = [], []
early_stopping_patience  = 15
epochs_without_improvement = 0

print(f"\nStarting Hybrid Training for {num_epochs} epochs...")
print("=" * 60)

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc     = evaluate(model, val_loader, loss_fn, device)

    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:02d}/{num_epochs}] | LR: {current_lr:.6f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, "hybrid_resnet_best.pth")
        print(f"  💾 Best model saved (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️ Early stopping triggered after {epoch+1} epochs.")
        break

    print("-" * 60)

# ─────────────────────────────────────────────
# FINAL TEST EVALUATION
# ─────────────────────────────────────────────
print("\nLoading best model for final test evaluation...")
checkpoint = torch.load("hybrid_resnet_best.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
print("=" * 60)
print(f"  Final Test Loss: {test_loss:.4f}")
print(f"  Final Test Acc:  {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Best Val Acc:    {best_val_acc:.4f}  ({best_val_acc*100:.2f}%)")
print("=" * 60)

Using device: cuda
Datasets loaded successfully
Class weights computed: [2.06858092 0.6736476  1.70988623 2.1727318  6.55413534 0.78025421
 0.33690191 0.69149611 2.01875868 1.55494113]
Total trainable parameters: 717,978

Starting Hybrid Training for 80 epochs...


Epoch [01/80] | LR: 0.000500
  Train Loss: 2.1516 | Train Acc: 0.2286
  Val   Loss: 2.0437 | Val   Acc: 0.6554
  💾 Best model saved (Val Acc: 0.6554)
------------------------------------------------------------


Epoch [02/80] | LR: 0.000499
  Train Loss: 1.6455 | Train Acc: 0.6276
  Val   Loss: 1.7363 | Val   Acc: 0.7804
  💾 Best model saved (Val Acc: 0.7804)
------------------------------------------------------------


Epoch [03/80] | LR: 0.000498
  Train Loss: 1.4566 | Train Acc: 0.7144
  Val   Loss: 1.6964 | Val   Acc: 0.7895
  💾 Best model saved (Val Acc: 0.7895)
------------------------------------------------------------


Epoch [04/80] | LR: 0.000497
  Train Loss: 1.3774 | Train Acc: 0.7462
  Val   Loss: 1.6465 | Val   Acc: 0.7935
  💾 Best model saved (Val Acc: 0.7935)
------------------------------------------------------------


...

Epoch [28/80] | LR: 0.000364
  Train Loss: 1.0855 | Train Acc: 0.8688
  Val   Loss: 1.4573 | Val   Acc: 0.8615
  🕒 No improvement for 4 epoch(s).
------------------------------------------------------------


Epoch [29/80] | LR: 0.000355
  Train Loss: 1.0802 | Train Acc: 0.8708
  Val   Loss: 1.4584 | Val   Acc: 0.8656
  🕒 No improvement for 5 epoch(s).
------------------------------------------------------------


Epoch [30/80] | LR: 0.000346
  Train Loss: 1.0752 | Train Acc: 0.8709
  Val   Loss: 1.4797 | Val   Acc: 0.8598
  🕒 No improvement for 6 epoch(s).
------------------------------------------------------------


Epoch [31/80] | LR: 0.000337
  Train Loss: 1.0752 | Train Acc: 0.8739
  Val   Loss: 1.4450 | Val   Acc: 0.8753
  🕒 No improvement for 7 epoch(s).
------------------------------------------------------------


KeyboardInterrupt: 